In [246]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [247]:
df = pd.read_csv("train.txt",sep =";",header=None,names=["text","emotion"])

In [248]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [249]:
unique_emotions=df["emotion"].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] =i
  i+=1

df["emotion"] = df["emotion"].map(emotion_numbers)

In [250]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [251]:
df["text"] =df['text'].apply(lambda x : x.lower())

In [252]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans("","",string.punctuation))

def remove_numbers(text):
    return text.translate(str.maketrans("","",string.digits))

In [253]:
df["text"] = df["text"].apply(remove_punctuation)
df["text"] = df["text"].apply(remove_numbers)

In [254]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new +=i
  return  new

In [255]:
df["text"] = df["text"].apply(remove_emojis)

In [256]:
import nltk

In [257]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [258]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

/home/anshad/anaconda3/envs/cufflinks-env/lib/python3.11/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/anshad/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt to /home/anshad/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/anshad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/anshad/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [259]:
stop_words = set(stopwords.words("english"))

In [260]:
df.loc[1]["text"]

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [261]:
# def remove(txt):
#   words = word_tokenize(txt)
#   return words

def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return " ".join(cleaned)

In [262]:
df["text"] = df["text"].apply(remove)

In [263]:
df.loc[1]

text       go feeling hopeless damned hopeful around some...
emotion                                                    0
Name: 1, dtype: object

In [264]:
from sklearn.model_selection import train_test_split

In [265]:
X_train, X_test, y_train, y_test = train_test_split(
   df["text"], df["emotion"], test_size=0.20, random_state=42)

In [269]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)



In [270]:
print(accuracy_score(y_test, pred_bow))

0.768125


In [271]:
tifidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tifidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tifidf_vectorizer.transform(X_test)


In [272]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)
pred_tfidf = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, pred_tfidf))

0.6609375


In [273]:
from sklearn.linear_model import LogisticRegression

In [278]:
lr_model = LogisticRegression(max_iter=1000) 
lr_model.fit(X_train_tfidf, y_train)
pred_lr = lr_model.predict(X_test_tfidf)

print(accuracy_score(y_test, pred_lr))

0.8628125


In [281]:

import pickle

# Save the trained model and TF-IDF vectorizer
artifacts = {
    "model": lr_model,
    "vectorizer": tifidf_vectorizer,
    "id_to_emotion": {
        int(number): str(emotion)
        for emotion, number in emotion_numbers.items()
    },
    "stop_words": set(stop_words)
}

with open("emotion_model.pkl", "wb") as file:
    pickle.dump(artifacts, file)

print("Model saved successfully!")


Model saved successfully!
